# U.S. Economy Walkthrough

This notebook pulls the current U.S. macro data used in the project, recreates the charts, and adds context for what each plot is saying about the economy right now.

Theme: the economy looks slower, not broken. Growth has cooled, labor is softening rather than collapsing, inflation is lower than its peak but still not fully settled, and policy still leans restrictive.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'scripts'))

from us_economy_analysis import fetch_series, build_monthly_frame, build_plot_style, compute_latest_points, SERIES

build_plot_style()
raw = {series_id: fetch_series(series_id) for series_id in SERIES}
monthly = build_monthly_frame(raw)
recent = monthly.loc[monthly.index >= (monthly.index.max() - pd.DateOffset(years=10))].copy()
latest_points = {point.name: point for point in compute_latest_points(monthly, raw)}

def month_end(ts):
    return pd.Timestamp(ts).to_period('M').to_timestamp('M')

latest_summary = pd.DataFrame(
    [
        ['Real GDP YoY', latest_points['Real GDP YoY'].date.strftime('Q%q %Y') if False else f"Q{((latest_points['Real GDP YoY'].date.month - 1) // 3) + 1} {latest_points['Real GDP YoY'].date.year}", monthly.loc[month_end(latest_points['Real GDP YoY'].date), 'gdp_yoy']],
        ['GDP QoQ Annualized', f"Q{((latest_points['Real GDP YoY'].date.month - 1) // 3) + 1} {latest_points['Real GDP YoY'].date.year}", monthly.loc[month_end(latest_points['Real GDP YoY'].date), 'gdp_qoq_annualized']],
        ['Unemployment Rate', latest_points['Unemployment rate'].date.strftime('%B %Y'), latest_points['Unemployment rate'].value],
        ['Payrolls 3M Avg Change', latest_points['Payroll growth, 3m avg'].date.strftime('%B %Y'), monthly.loc[month_end(latest_points['Payroll growth, 3m avg'].date), 'payrolls_3m_avg_change'] * 1000],
        ['CPI YoY', latest_points['CPI inflation YoY'].date.strftime('%B %Y'), monthly.loc[month_end(latest_points['CPI inflation YoY'].date), 'cpi_yoy']],
        ['CPI 3M Annualized', latest_points['CPI inflation YoY'].date.strftime('%B %Y'), monthly.loc[month_end(latest_points['CPI inflation YoY'].date), 'cpi_3m_ann']],
        ['Core PCE YoY', latest_points['Core PCE inflation YoY'].date.strftime('%B %Y'), monthly.loc[month_end(latest_points['Core PCE inflation YoY'].date), 'core_pce_yoy']],
        ['Core PCE 3M Annualized', latest_points['Core PCE inflation YoY'].date.strftime('%B %Y'), monthly.loc[month_end(latest_points['Core PCE inflation YoY'].date), 'core_pce_3m_ann']],
        ['Fed Funds', latest_points['Fed funds rate'].date.strftime('%B %Y'), latest_points['Fed funds rate'].value],
        ['10Y-2Y Spread', latest_points['10Y-2Y spread'].date.strftime('%B %d, %Y'), latest_points['10Y-2Y spread'].value],
        ['Industrial Production YoY', latest_points['Industrial production YoY'].date.strftime('%B %Y'), latest_points['Industrial production YoY'].value],
        ['Real Retail Sales YoY', latest_points['Real retail sales YoY'].date.strftime('%B %Y'), latest_points['Real retail sales YoY'].value],
    ],
    columns=['Indicator', 'As Of', 'Value']
)

latest_summary

## Where We Are Right Now

A useful way to frame the current U.S. economy is: still expanding, but with less momentum and less margin for error than a year ago. Growth has slowed, labor is cooler, and inflation is lower than at its peak, but the last few months still show enough firmness that the Federal Reserve cannot declare the job done.

In [ ]:
gdp_date = month_end(latest_points['Real GDP YoY'].date)
labor_date = month_end(latest_points['Payroll growth, 3m avg'].date)
infl_date = month_end(latest_points['CPI inflation YoY'].date)
core_pce_date = month_end(latest_points['Core PCE inflation YoY'].date)
real_policy = monthly['real_policy_rate_proxy'].dropna().iloc[-1]

display(Markdown(
    f"""
- **Growth:** real GDP was **{monthly.loc[gdp_date, 'gdp_yoy']:.2f}% YoY** in the latest quarter, but the most recent quarterly pace slowed to **{monthly.loc[gdp_date, 'gdp_qoq_annualized']:.2f}% annualized**.
- **Labor:** unemployment was **{latest_points['Unemployment rate'].value:.1f}%** and payroll growth averaged about **{monthly.loc[labor_date, 'payrolls_3m_avg_change'] * 1000:,.0f} jobs/month** over the latest three months.
- **Inflation:** CPI was **{monthly.loc[infl_date, 'cpi_yoy']:.2f}% YoY** while core PCE was **{monthly.loc[core_pce_date, 'core_pce_yoy']:.2f}% YoY**.
- **Policy:** the fed funds rate was **{latest_points['Fed funds rate'].value:.2f}%**, implying a rough real policy rate of **{real_policy:.2f}%**.

That combination is consistent with a late-cycle economy that is cooling, but not yet giving a classic recession signal.
"""
))

## Growth And Activity

This plot compares real GDP growth with industrial production growth. GDP is broad and usually smoother; industrial production is narrower and more cyclical. When GDP is still positive but industrial production is only modestly positive, that usually points to an economy that is still expanding but with less help from manufacturing and other goods-producing sectors.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(recent.index, recent['gdp_yoy'], label='Real GDP YoY', linewidth=2.5, color='#355c7d')
ax.plot(recent.index, recent['industrial_production_yoy'], label='Industrial Production YoY', linewidth=2.5, color='#c06c84')
ax.axhline(0, color='#403a2f', linewidth=1.2, linestyle='--', alpha=0.75)
ax.set_title('Growth And Activity')
ax.set_ylabel('Percent change')
ax.grid(True, linestyle=':', alpha=0.7)
ax.legend(frameon=False)
plt.show()

display(Markdown(
    f"""
**Current read:** GDP is running at **{monthly.loc[gdp_date, 'gdp_yoy']:.2f}% YoY**, but the latest quarterly annualized pace is only **{monthly.loc[gdp_date, 'gdp_qoq_annualized']:.2f}%**. Industrial production is **{latest_points['Industrial production YoY'].value:.2f}% YoY**. That mix usually means the economy is still growing, but the cyclical sectors are not powering a broad acceleration.
"""
))

## Labor Market

This plot puts the unemployment rate next to the three-month average of payroll changes. The unemployment rate tells us about slack; payroll growth tells us about hiring momentum. It is normal for payroll growth to cool before unemployment rises sharply, so the combination often matters more than either series alone.

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 6))
ax2 = ax1.twinx()
ax1.plot(recent.index, recent['UNRATE'], label='Unemployment Rate', linewidth=2.5, color='#355c7d')
ax2.plot(recent.index, recent['payrolls_3m_avg_change'] * 1000, label='Payrolls 3M Avg Change', linewidth=2.5, color='#f67280')
ax1.set_title('Labor Market')
ax1.set_ylabel('Unemployment rate (%)', color='#355c7d')
ax2.set_ylabel('Jobs per month', color='#f67280')
ax1.grid(True, linestyle=':', alpha=0.7)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, frameon=False, loc='upper left')
plt.show()

display(Markdown(
    f"""
**Current read:** unemployment is **{latest_points['Unemployment rate'].value:.1f}%**, still low in historical terms, but off its recent floor. Payroll growth has slowed to roughly **{monthly.loc[labor_date, 'payrolls_3m_avg_change'] * 1000:,.0f} jobs/month** on a three-month average basis. That looks more like gradual cooling than a sudden labor-market break.
"""
))

## Inflation

This plot shows both year-over-year and three-month annualized inflation for CPI and core PCE. The year-over-year rate captures the broad disinflation trend; the three-month annualized rate is better for judging whether the latest few months are running hotter or cooler than that trend.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(recent.index, recent['cpi_yoy'], label='CPI YoY', linewidth=2.5, color='#355c7d')
ax.plot(recent.index, recent['core_pce_yoy'], label='Core PCE YoY', linewidth=2.5, color='#c06c84')
ax.plot(recent.index, recent['cpi_3m_ann'], label='CPI 3M Annualized', linewidth=2.5, color='#f67280')
ax.plot(recent.index, recent['core_pce_3m_ann'], label='Core PCE 3M Annualized', linewidth=2.5, color='#6c5b7b')
ax.axhline(2, color='#403a2f', linewidth=1.2, linestyle='--', alpha=0.6)
ax.set_title('Inflation')
ax.set_ylabel('Percent')
ax.grid(True, linestyle=':', alpha=0.7)
ax.legend(frameon=False)
plt.show()

display(Markdown(
    f"""
**Current read:** CPI is **{monthly.loc[infl_date, 'cpi_yoy']:.2f}% YoY** and core PCE is **{monthly.loc[core_pce_date, 'core_pce_yoy']:.2f}% YoY**. But the shorter-run measures are **{monthly.loc[infl_date, 'cpi_3m_ann']:.2f}%** for CPI and **{monthly.loc[core_pce_date, 'core_pce_3m_ann']:.2f}%** for core PCE. Because those short-run numbers are firmer than the 12-month trend, the story is not 'inflation solved' so much as 'inflation improved, but still uneven.'
"""
))

## Rates And The Curve

This plot combines the fed funds rate, the 10Y-2Y Treasury spread, and a simple real policy rate proxy. The fed funds rate tells us the policy stance, the curve reflects market expectations about growth and future cuts, and the real policy rate tells us whether policy is still restrictive after adjusting for inflation.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(recent.index, recent['FEDFUNDS'], label='Fed Funds', linewidth=2.5, color='#355c7d')
ax.plot(recent.index, recent['T10Y2Y'], label='10Y-2Y Spread', linewidth=2.5, color='#c06c84')
ax.plot(recent.index, recent['real_policy_rate_proxy'], label='Real Policy Rate Proxy', linewidth=2.5, color='#f67280')
ax.axhline(0, color='#403a2f', linewidth=1.2, linestyle='--', alpha=0.75)
ax.set_title('Rates And Curve')
ax.set_ylabel('Percent / percentage points')
ax.grid(True, linestyle=':', alpha=0.7)
ax.legend(frameon=False)
plt.show()

display(Markdown(
    f"""
**Current read:** the fed funds rate is **{latest_points['Fed funds rate'].value:.2f}%**, the 10Y-2Y spread is **{latest_points['10Y-2Y spread'].value:.2f} percentage points**, and the real policy rate proxy is **{real_policy:.2f}%**. That suggests policy is still leaning against demand, but not with the same intensity as when inflation was much higher and the curve was deeply inverted.
"""
))

## Consumer And Housing

This plot brings together real retail sales growth and housing starts growth. Consumption tells us whether households are still carrying the expansion; housing tells us how the most rate-sensitive part of the economy is responding. The combination is useful because weak housing does not automatically mean recession if consumers are still spending in real terms.

In [ ]:
housing_date = monthly['housing_starts_yoy'].dropna().index[-1]
retail_date = month_end(latest_points['Real retail sales YoY'].date)

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(recent.index, recent['real_retail_sales_yoy'], label='Real Retail Sales YoY', linewidth=2.5, color='#355c7d')
ax.plot(recent.index, recent['housing_starts_yoy'], label='Housing Starts YoY', linewidth=2.5, color='#c06c84')
ax.axhline(0, color='#403a2f', linewidth=1.2, linestyle='--', alpha=0.75)
ax.set_title('Consumer And Housing')
ax.set_ylabel('Percent change')
ax.grid(True, linestyle=':', alpha=0.7)
ax.legend(frameon=False)
plt.show()

display(Markdown(
    f"""
**Current read:** real retail sales are **{monthly.loc[retail_date, 'real_retail_sales_yoy']:.2f}% YoY**, while housing starts are **{monthly.loc[housing_date, 'housing_starts_yoy']:.2f}% YoY**. That combination says households are still spending, but the most interest-rate-sensitive sectors remain volatile. It is a mixed late-cycle picture, not a clean boom and not a clean bust.
"""
))

## Takeaways

- The economy still looks expansionary, but growth has lost speed.
- Labor market cooling is real, though the data do not yet point to a sharp downturn.
- Inflation has improved materially from its highs, but the latest few months still argue for caution.
- Monetary policy still appears somewhat restrictive, which helps explain the softer tone in rate-sensitive sectors.
- The most defensible interpretation right now is a cooling late-cycle economy with mixed but not yet recessionary signals.